# Single-Camera Tracking Consistency — LOCAL runner (NVIDIA GPU)

Same cross-scene experiment as the Colab notebook, but for a **local machine with an NVIDIA GPU**.
Trains the learned ReID matcher on one warehouse, tests it on a **different, unseen** one, and saves a clean
`comparison.md` (raw vs conservative vs learned) to **local disk**.

**Before running:** make sure your Python environment has a **CUDA-enabled `torch` + `torchvision`** already
installed (see the setup cell below for the rest — it installs everything else from the repo's `requirements.txt`
files). Paste your HuggingFace token in the Environment cell.

---
## Environment

In [ ]:
import os, sys, shutil, subprocess, glob
# ===== EDIT THESE =====
REPO     = os.path.abspath('Single-Camera-Tracking-Consistency')  # repo cloned/updated here
PY       = sys.executable     # this kernel's python — must have CUDA torch + BoT-SORT/torchreid
HF_TOKEN = ''                 # <-- paste your HuggingFace token here
# ======================
ON_COLAB = False; DRIVE = None
print('REPO   :', REPO)
print('python :', PY)
print('HF set :', bool(HF_TOKEN))

---
## Clone / update the repo (branch: main)
Clones the repo into `REPO` if missing, otherwise pulls the latest `main`.

In [ ]:
BRANCH  = 'main'
GIT_URL = 'https://github.com/Hithesh18/Single-Camera-Tracking-Consistency.git'
if not os.path.isdir(f'{REPO}/.git'):
    print(f'cloning {BRANCH} -> {REPO} ...', flush=True)
    subprocess.run(['git','clone','--branch',BRANCH,GIT_URL,REPO], check=True)
else:
    subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=False)
    subprocess.run(['git','-C',REPO,'clean','-fd','tracklet_repair/models','tracklet_repair/results'], check=False)
    subprocess.run(['git','-C',REPO,'checkout','-f',BRANCH], check=False)
    subprocess.run(['git','-C',REPO,'reset','--hard',f'origin/{BRANCH}'], check=False)
os.chdir(REPO)
assert os.path.isdir(f'{REPO}/tracklet_repair'), 'repo missing tracklet_repair — clone failed?'
try:
    import torch; print('repo ready on', BRANCH, '| CUDA GPU:', torch.cuda.is_available(), '(need True)')
except Exception as e:
    print('repo ready on', BRANCH, '| torch not importable yet:', e)

---
## One-time local setup (skip if already done)
Please install the necessary packages before running anything else. This installs from the repo's own
`requirements.txt` files (`BoT-SORT/`, `deep-person-reid/`, `tracklet_repair/`, `tracking/`), the extra packages
those don't cover, registers BoT-SORT / torchreid as importable via `setup.py develop`, and downloads the two
model checkpoints (OSNet ReID, ByteTrack detector) if they aren't already on disk.

Your env must already have a **CUDA-enabled `torch` + `torchvision`** installed separately (pick the build for
your CUDA version from [pytorch.org](https://pytorch.org/get-started/locally/) — this differs per machine, so it
is not in any `requirements.txt`).</cell id="cell-5">


In [ ]:
setup_marker = f'{REPO}/.local_deps_ok'
if os.path.exists(setup_marker):
    print('Dependencies already installed — skipping (delete .local_deps_ok to force a re-install).')
else:
    # requirements.txt files already in the repo
    for req in ['BoT-SORT/requirements.txt', 'deep-person-reid/requirements.txt',
                'tracklet_repair/requirements.txt', 'tracking/requirements.txt']:
        req_path = f'{REPO}/{req}'
        if os.path.exists(req_path):
            print(f'installing from {req} ...')
            subprocess.run([PY, '-m', 'pip', 'install', '-q', '-r', req_path], check=False)

    # extra packages none of the requirements.txt above cover
    for _p in ['cython_bbox', 'pycocotools', 'huggingface_hub']:
        subprocess.run([PY, '-m', 'pip', 'install', '-q', _p], check=False)
    if subprocess.run([PY, '-m', 'pip', 'install', '-q', 'faiss-gpu'], capture_output=True).returncode != 0:
        subprocess.run([PY, '-m', 'pip', 'install', '-q', 'faiss-cpu'], check=False)

    # register BoT-SORT / torchreid as importable packages
    for pkg in ['BoT-SORT', 'deep-person-reid']:
        if os.path.isdir(f'{REPO}/{pkg}'):
            subprocess.run([PY, 'setup.py', 'develop', '--quiet'], cwd=f'{REPO}/{pkg}', check=False)

    open(setup_marker, 'w').write('ok\n')

os.chdir(REPO)

# model checkpoints (only fetched if missing — safe to re-run)
osnet_local = f'{REPO}/deep-person-reid/checkpoints/osnet_ms_m_c.pth.tar'
if not os.path.exists(osnet_local):
    print('downloading OSNet checkpoint from the HuggingFace mirror ...')
    from huggingface_hub import hf_hub_download
    os.makedirs(os.path.dirname(osnet_local), exist_ok=True)
    fn = 'osnet_x1_0_msmt17_combineall_256x128_amsgrad_ep150_stp60_lr0.0015_b64_fb10_softmax_labelsmooth_flip_jitter.pth'
    src = hf_hub_download(repo_id='kaiyangzhou/osnet', filename=fn)
    shutil.copy(src, osnet_local)
    print('OSNet: downloaded ->', os.path.getsize(osnet_local), 'bytes')
else:
    print('OSNet: already local')

bt_local = f'{REPO}/BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar'
if not os.path.exists(bt_local):
    print('downloading ByteTrack checkpoint ...')
    os.makedirs(os.path.dirname(bt_local), exist_ok=True)
    subprocess.run([PY, '-m', 'pip', 'install', '-q', '-U', 'gdown'], check=False)
    import gdown
    gdown.download(id='1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5', output=bt_local, quiet=False)
else:
    print('ByteTrack: already local')

print('setup done.')

---
## Configuration

In [ ]:
# ===== CONFIGURATION — edit ONLY here =====
DATASET      = 'Val'
TRAIN_SCENE  = 'Warehouse_016'   # the matcher LEARNS from this scene
TEST_SCENE   = 'Warehouse_015'   # the matcher is TESTED on this (unseen) scene
CAMERAS      = ['Camera','Camera_01','Camera_02','Camera_03']   # None = all 12
MAXF         = 1500              # frames per camera (0 = all 9000)
TRACK_PARAMS = {}                # BoT-SORT overrides, e.g. {'match_thresh':0.9}
RESULTS_DIR  = f'{REPO}/results_local'   # clean records saved here (local disk)
os.chdir(REPO)
assert TRAIN_SCENE != TEST_SCENE, 'TRAIN and TEST scenes must differ (never train+test the same scene).'
print(f'TRAIN on {TRAIN_SCENE}  ->  TEST on {TEST_SCENE} | cams {CAMERAS or "ALL"} | cap {MAXF or "ALL"}')

---
## Build the runner  *(all logic lives in tracklet_repair/src/pipeline/cross_scene_runner.py)*

In [ ]:
from tracklet_repair.src.pipeline.cross_scene_runner import CrossSceneRunner
runner = CrossSceneRunner(repo=REPO, py=PY, dataset=DATASET, cameras=CAMERAS, maxf=MAXF,
                          track_params=TRACK_PARAMS, hf_token=HF_TOKEN,
                          results_dir=RESULTS_DIR, on_colab=False)
print('runner ready')

---
## Step 1 — Download TRAIN scene (videos + ground truth)

In [ ]:
runner.download_scene(TRAIN_SCENE)

## Step 2 — Single-camera tracking, TRAIN scene  *(GPU; the long step)*

In [ ]:
runner.generate_scene(TRAIN_SCENE)

## Step 3 — Train the matcher  *(on TRAIN scene; CPU, fast)*

In [ ]:
runner.train_on(TRAIN_SCENE)

## Step 4 — Download TEST scene (videos + ground truth)

In [ ]:
runner.download_scene(TEST_SCENE)

## Step 5 — Single-camera tracking, TEST scene  *(GPU; the long step)*

In [ ]:
runner.generate_scene(TEST_SCENE)

## Step 6 — Test + save results  *(CPU; applies TRAIN matcher to unseen TEST scene)*

In [ ]:
runner.benchmark(TEST_SCENE, TRAIN_SCENE)